<a href="https://colab.research.google.com/github/hassebcs2024/BSCS-2024-AI-Labs/blob/main/Lab1_KBS_ExpertSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Artificial Intelligence — Practical Lab 1
## Google Colab Setup and Knowledge-Based System (KBS)

## Knowledge-Based System (KBS)

In my system, the **Knowledge Base** stores rules, and the **Inference Engine** checks student details using those rules.

## Part 1 — Google Colab Setup

I am using Google Colab to run my Python program. First, I will check that it is working.

In [2]:
print("Google Colab Environment Initialized Successfully!")

Google Colab Environment Initialized Successfully!


## Task 1 — Knowledge Base

Now I will create the **Knowledge Base** and add the six university rules for attendance and grades. I have stored these rules in `UNIVERSITY_KB`.

In [4]:
# TASK 1: KNOWLEDGE BASE — the rules provided in the lab.
UNIVERSITY_KB = {
    "system_name": "BSCS Academic Performance & Grading Expert System",
    "passing_marks": 50,
    "rules": [
        {
            "id": "RULE_01",
            "conditions": {"attendance_percentage": 74},  # 74% or less
            "condition_type": "attendance_below",
            "grade": "F",
            "status": "Debarred",
            "action": "Ineligible to sit for final exam due to short attendance."
        },
        {
            "id": "RULE_02",
            "conditions": {"marks_min": 85, "marks_max": 100},
            "condition_type": "marks_range",
            "grade": "A",
            "status": "Passed with Distinction",
            "action": "Recommended for Merit Scholarship."
        },
        {
            "id": "RULE_03",
            "conditions": {"marks_min": 75, "marks_max": 84},
            "condition_type": "marks_range",
            "grade": "B",
            "status": "Passed",
            "action": "Good standing. Maintain momentum."
        },
        {
            "id": "RULE_04",
            "conditions": {"marks_min": 60, "marks_max": 74},
            "condition_type": "marks_range",
            "grade": "C",
            "status": "Passed",
            "action": "Satisfactory performance. Focus on weak subject areas."
        },
        {
            "id": "RULE_05",
            "conditions": {"marks_min": 50, "marks_max": 59},
            "condition_type": "marks_range",
            "grade": "D",
            "status": "Conditional Pass",
            "action": "Warning: Near failure threshold. Academic counseling advised."
        },
        {
            "id": "RULE_06",
            "conditions": {"marks_min": 0, "marks_max": 49},
            "condition_type": "marks_range",
            "grade": "F",
            "status": "Failed",
            "action": "Must repeat the course in the upcoming semester."
        }
    ]
}

print(f"Loaded: {UNIVERSITY_KB['system_name']}")
print(f"Original grading rules: {len(UNIVERSITY_KB['rules'])}")

Loaded: BSCS Academic Performance & Grading Expert System
Original grading rules: 6


## Exercise 1 — Add Honors Rule

I added **RULE_07** for graduating students. A student gets honors if the degree is complete, CGPA is **3.70 or above**, and there are **no warnings**.

In [ ]:
# EXERCISE 1: ADD ONE NEW RULE to the same Knowledge Base.
UNIVERSITY_KB["rules"].append({
    "id": "RULE_07",
    "conditions": {
        "degree_completed": True,
        "cgpa_min": 3.70,
        "disciplinary_warnings": 0
    },
    "condition_type": "degree_honors",
    "status": "Graduated with Magna Cum Laude Honors",
    "action": "Congratulations! Eligible for Magna Cum Laude Honors."
})

print("New rule added:", UNIVERSITY_KB["rules"][-1]["id"])
print("Total number of rules:", len(UNIVERSITY_KB["rules"]))

New rule added: RULE_07
Total number of rules: 7


## Task 2 — Inference Engine and Exercise 2 — Rule Tracing

Now I will make the **Inference Engine** to check the rules and give each student's result. I also made it print **True** or **False** for every rule, so I can see how it makes its decision.

In [5]:
# TASK 2 + EXERCISE 2: INFERENCE ENGINE WITH STEP-BY-STEP TRACING.
def evaluate_student(student_record, kb):
    """Check all university rules and return a student's final result."""
    print(f"\n--- EVALUATING STUDENT: {student_record['student_name']} "
          f"(ID: {student_record['student_id']}) ---")

    attendance_rule = None  # Highest priority when attendance is short
    marks_rule = None       # Rule for the student's course marks
    honors_rule = None      # Optional rule for completed degree

    # Check the conditions of EACH rule and print what happened.
    for rule in kb["rules"]:
        kind = rule["condition_type"]

        if kind == "attendance_below":
            attendance = student_record["attendance"]
            threshold = rule["conditions"]["attendance_percentage"] + 1
            matched = attendance < threshold
            print(f"{rule['id']}: {attendance}% attendance < {threshold}%? {matched}")
            if matched:
                attendance_rule = rule

        elif kind == "marks_range":
            marks = student_record["total_marks"]
            low = rule["conditions"]["marks_min"]
            high = rule["conditions"]["marks_max"]
            matched = low <= marks <= high
            print(f"{rule['id']}: {low} <= {marks} marks <= {high}? {matched}")
            if matched:
                marks_rule = rule

        elif kind == "degree_honors":
            completed = student_record.get("degree_completed", False)
            cgpa = student_record.get("cgpa", 0)
            warnings = student_record.get("disciplinary_warnings", 0)
            required = rule["conditions"]
            matched = (completed == required["degree_completed"]
                       and cgpa >= required["cgpa_min"]
                       and warnings == required["disciplinary_warnings"])
            print(f"{rule['id']}: degree completed={completed}, CGPA={cgpa}, "
                  f"warnings={warnings}; honors conditions satisfied? {matched}")
            if matched:
                honors_rule = rule

    # Attendance takes priority over course marks.
    chosen_rule = attendance_rule if attendance_rule is not None else marks_rule

    if chosen_rule is None:
        result = {
            "rule_id": "UNKNOWN",
            "grade": "N/A",
            "status": "Invalid Score Entry",
            "action": "Verify submitted marks with the exam department."
        }
    else:
        result = {
            "rule_id": chosen_rule["id"],
            "grade": chosen_rule["grade"],
            "status": chosen_rule["status"],
            "action": chosen_rule["action"]
        }

    # Keep the course status and check graduation honors separately.
    result["course_status"] = result["status"]
    result["course_action"] = result["action"]
    result["honors_status"] = "Not awarded"
    if honors_rule is not None and chosen_rule is not None:
        result["honors_status"] = honors_rule["status"]
        result["status"] = honors_rule["status"]  # Exercise 1: exact status
        result["action"] = honors_rule["action"]
        result["honors_rule_id"] = honors_rule["id"]

    print(f"FINAL RESULT -> Grade: {result['grade']} | Status: {result['status']}")
    print(f"Advisory Action: {result['action']}")
    return result

print("Inference Engine is ready!")

Inference Engine is ready!


## Task 3 — Student Records

Now I will test my program using **Ali Khan** and **Sara Ahmed** from the lab. Ali should get **A**, while Sara should be **Debarred** because of low attendance.

In [ ]:
# TASK 3: ORIGINAL TWO STUDENTS.
student_1 = {
    "student_id": "BSCS-2024-012",
    "student_name": "Ali Khan",
    "total_marks": 88,
    "attendance": 92
}
student_2 = {
    "student_id": "BSCS-2024-045",
    "student_name": "Sara Ahmed",
    "total_marks": 81,
    "attendance": 68
}

result_1 = evaluate_student(student_1, UNIVERSITY_KB)
result_2 = evaluate_student(student_2, UNIVERSITY_KB)

print("\nSUMMARY OF TASK 3")
print("Ali  ->", result_1["grade"], "|", result_1["status"])
print("Sara ->", result_2["grade"], "|", result_2["status"])


--- EVALUATING STUDENT: Ali Khan (ID: BSCS-2024-012) ---
RULE_01: 92% attendance < 75%? False
RULE_02: 85 <= 88 marks <= 100? True
RULE_03: 75 <= 88 marks <= 84? False
RULE_04: 60 <= 88 marks <= 74? False
RULE_05: 50 <= 88 marks <= 59? False
RULE_06: 0 <= 88 marks <= 49? False
RULE_07: degree completed=False, CGPA=0, warnings=0; honors conditions satisfied? False
FINAL RESULT -> Grade: A | Status: Passed with Distinction
Advisory Action: Recommended for Merit Scholarship.

--- EVALUATING STUDENT: Sara Ahmed (ID: BSCS-2024-045) ---
RULE_01: 68% attendance < 75%? True
RULE_02: 85 <= 81 marks <= 100? False
RULE_03: 75 <= 81 marks <= 84? True
RULE_04: 60 <= 81 marks <= 74? False
RULE_05: 50 <= 81 marks <= 59? False
RULE_06: 0 <= 81 marks <= 49? False
RULE_07: degree completed=False, CGPA=0, warnings=0; honors conditions satisfied? False
FINAL RESULT -> Grade: F | Status: Debarred
Advisory Action: Ineligible to sit for final exam due to short attendance.

SUMMARY OF TASK 3
Ali  -> A | Pass

## Exercise 1 — Test Honors Rule

Now I will test the honors rule using **Fatima, Bilal, and Ayesha**. Fatima should get honors, but Bilal and Ayesha should not.

In [ ]:
# EXTRA TEST STUDENTS FOR EXERCISE 1.
student_3 = {
    "student_id": "BSCS-2024-078",
    "student_name": "Fatima",
    "total_marks": 80,
    "attendance": 95,
    "degree_completed": True,
    "cgpa": 3.85,
    "disciplinary_warnings": 0
}
student_4 = {
    "student_id": "BSCS-2024-079",
    "student_name": "Bilal",
    "total_marks": 90,
    "attendance": 94,
    "degree_completed": True,
    "cgpa": 3.85,
    "disciplinary_warnings": 1
}
student_5 = {
    "student_id": "BSCS-2024-080",
    "student_name": "Ayesha",
    "total_marks": 76,
    "attendance": 90,
    "degree_completed": True,
    "cgpa": 3.69,
    "disciplinary_warnings": 0
}

result_3 = evaluate_student(student_3, UNIVERSITY_KB)
result_4 = evaluate_student(student_4, UNIVERSITY_KB)
result_5 = evaluate_student(student_5, UNIVERSITY_KB)

print("\nHONORS EXERCISE SUMMARY")
for student, result in [(student_3, result_3), (student_4, result_4),
                        (student_5, result_5)]:
    print(f"{student['student_name']}: grade={result['grade']}; "
          f"honors={result['honors_status']}")


--- EVALUATING STUDENT: Fatima (ID: BSCS-2024-078) ---
RULE_01: 95% attendance < 75%? False
RULE_02: 85 <= 80 marks <= 100? False
RULE_03: 75 <= 80 marks <= 84? True
RULE_04: 60 <= 80 marks <= 74? False
RULE_05: 50 <= 80 marks <= 59? False
RULE_06: 0 <= 80 marks <= 49? False
RULE_07: degree completed=True, CGPA=3.85, warnings=0; honors conditions satisfied? True
FINAL RESULT -> Grade: B | Status: Graduated with Magna Cum Laude Honors
Advisory Action: Congratulations! Eligible for Magna Cum Laude Honors.

--- EVALUATING STUDENT: Bilal (ID: BSCS-2024-079) ---
RULE_01: 94% attendance < 75%? False
RULE_02: 85 <= 90 marks <= 100? True
RULE_03: 75 <= 90 marks <= 84? False
RULE_04: 60 <= 90 marks <= 74? False
RULE_05: 50 <= 90 marks <= 59? False
RULE_06: 0 <= 90 marks <= 49? False
RULE_07: degree completed=True, CGPA=3.85, warnings=1; honors conditions satisfied? False
FINAL RESULT -> Grade: A | Status: Passed with Distinction
Advisory Action: Recommended for Merit Scholarship.

--- EVALUATI

## Extra Tests

I also tested the grade limits, attendance limit, and CGPA limit. These tests help me check that my program gives the correct results.

In [ ]:
# Automatic verification (trace output is hidden here to keep tests readable).
from contextlib import redirect_stdout
from io import StringIO

def quiet_evaluate(student):
    with redirect_stdout(StringIO()):
        return evaluate_student(student, UNIVERSITY_KB)

# Grade-band endpoints and invalid marks.
for marks, expected_grade in [
    (100, "A"), (85, "A"), (84, "B"), (75, "B"),
    (74, "C"), (60, "C"), (59, "D"), (50, "D"),
    (49, "F"), (0, "F"), (101, "N/A"), (-1, "N/A")
]:
    record = {
        "student_id": "TEST", "student_name": "Test Student",
        "total_marks": marks, "attendance": 90
    }
    assert quiet_evaluate(record)["grade"] == expected_grade

# Attendance cutoff: 74% debarred; 75% allowed to receive a grade.
record = {"student_id": "TEST", "student_name": "Test",
          "total_marks": 90, "attendance": 74}
assert quiet_evaluate(record)["status"] == "Debarred"
record["attendance"] = 75
assert quiet_evaluate(record)["grade"] == "A"

# Honors cutoff and warnings/degree-completion checks.
record = {"student_id": "TEST", "student_name": "Test Graduate",
          "total_marks": 88, "attendance": 90, "degree_completed": True,
          "cgpa": 3.70, "disciplinary_warnings": 0}
assert quiet_evaluate(record)["honors_status"] == "Graduated with Magna Cum Laude Honors"
record["cgpa"] = 3.69
assert quiet_evaluate(record)["honors_status"] == "Not awarded"
record["cgpa"] = 3.85
record["disciplinary_warnings"] = 1
assert quiet_evaluate(record)["honors_status"] == "Not awarded"
record["disciplinary_warnings"] = 0
record["degree_completed"] = False
assert quiet_evaluate(record)["honors_status"] == "Not awarded"

print("All tests passed! Grade ranges, attendance, and honors work correctly.")

All tests passed! Grade ranges, attendance, and honors work correctly.


## Answers to the Exercises

**Exercise 1:** I added **RULE_07** to give honors when the degree is complete, CGPA is at least **3.70**, and there are **no warnings**.

**Exercise 2:** I made the Inference Engine print **True** or **False** for each rule, so I can see how it decides the result.

## Conclusion

I completed the Knowledge Base, Inference Engine, student tests, and both exercises in Google Colab.